# Module 03a — Curating SFT Data

**The thesis.** Most large public SFT datasets you'll find on HuggingFace for languages other than English are *aggregations of machine-translated English instruction data*. The example we work with here, `DeepMount00/OpenItalianData`, is no exception — 2.14M rows of mostly-translated content. There are real signal here. There is also a lot of translation noise: untranslated chunks, US-centric entities, calque phrasings that no native speaker would produce, and brand mentions of the model that originally generated the English target.

**The lesson.** Curation is where SFT quality is actually decided. LIMA showed that 1k high-quality examples can outperform 50k mediocre ones. Tülu 3's data ablations confirmed this at scale. A 100k high-precision Italian SFT set will beat a 2.14M noisy one — and the difference is the filter cascade, not the model.

This notebook walks through the 5-stage filter cascade implemented in `filters.py`. Each stage is cheap-to-expensive, so you cut the noise floor for free before paying for the next layer. At the end you'll see the full pipeline run end-to-end on a tiny synthetic fixture, with a drop-reason report — which is the artifact you should produce on your own runs and *actually look at*.

## Setup

This notebook runs offline on CPU. The MinHash + diversity-downsample cells optionally require `datasketch` and `sentence-transformers` — they degrade gracefully if not installed. The langid cell requires `fasttext-wheel` and downloads a ~1 MB model on first use; it's also gracefully optional.

Install the full set with `pip install -r requirements.txt` from this directory.

In [1]:
import sys
from pathlib import Path

# Make filters/judge importable from the module directory.
sys.path.insert(0, str(Path.cwd()))

from filters import (
    ArtifactConfig, LangConfig, StructuralConfig,
    calque_filter, empty_or_copy_filter, english_artifact_filter,
    function_word_filter, function_word_ratio, langid_filter,
    length_filter, ngram_repetition_score, repetition_filter,
    run_per_row_filters, user_text, assistant_text,
)

def row(user, assistant):
    return {'messages': [
        {'role': 'user', 'content': user},
        {'role': 'assistant', 'content': assistant},
    ]}

# A tiny synthetic fixture mixing legitimate Italian rows with each kind of
# noise we want to filter out. In real use this is a slice of OpenItalianData.
fixture = [
    # CLEAN — should survive every stage.
    row('Riassumi questo testo in due frasi: il sole è una stella della nostra galassia.',
        'Il sole è una stella che fornisce energia al sistema solare. Si trova a circa 150 milioni di km dalla Terra.'),
    row('Scrivi una breve poesia sull autunno e le foglie dorate.',
        'Cadono lente le foglie dorate, danzano leggere nel vento freddo, l autunno avvolge tutto in un velo di malinconia.'),
    row('Spiega cosa fa la funzione print() in Python.',
        'La funzione print() in Python stampa a video il contenuto degli argomenti che riceve, separati da uno spazio.'),
    # NOISE — each row triggers a different filter.
    row('Hi', 'OK'),                                                          # too short
    row('Genera una frase di esempio.', 'Genera una frase di esempio.'),     # copy bug
    row('Ripeti.', ' '.join(['test'] * 30)),                                  # repetition
    row('Cosa pensi della politica?', 'I\'m sorry, as an AI language model, I cannot answer that.'),  # untranslated
    row('Quanto costa una pizza a New York?', 'Una pizza media costa $25 nelle pizzerie di Manhattan.'),  # US dollar
    row('Qual è la tua opinione?', 'Come modello linguistico, non posso fornire opinioni personali soggettive.'),  # calque
    row('Traduci.', 'The cat is on the table eating fish under the moon and stars.'),  # English assistant
]
print(f'fixture: {len(fixture)} rows ({len(fixture) - 7} clean, 7 noise)')

fixture: 10 rows (3 clean, 7 noise)


## Stage 1 — Structural

The cheapest filters: pure-Python heuristics on length and repetition. These cut empties, copy bugs, and degenerate outputs without looking at content. Always run them first.

In [2]:
# Show what each structural filter catches on the fixture.
print(f'{"filter":<22} {"verdict":<40} prompt')
print('-' * 100)
for r in fixture:
    for fname, f in [
        ('empty_or_copy', empty_or_copy_filter),
        ('length',         length_filter),
        ('repetition',     repetition_filter),
    ]:
        verdict = f(r)
        if verdict is not None:
            print(f'{fname:<22} {verdict:<40} {user_text(r)[:55]!r}')
            break

filter                 verdict                                  prompt
----------------------------------------------------------------------------------------------------
length                 user_too_short                           'Hi'
empty_or_copy          response_equals_prompt                   'Genera una frase di esempio.'
length                 user_too_short                           'Ripeti.'
length                 assistant_too_short                      'Quanto costa una pizza a New York?'
length                 assistant_too_short                      'Qual è la tua opinione?'
length                 user_too_short                           'Traduci.'


In [3]:
# N-gram repetition score: pedagogical demo of what 'degenerate' looks like.
demos = [
    ('clean prose',  'Il sole illumina il mondo, le stelle brillano nel cielo notturno e la luna splende silenziosa.'),
    ('mild repeat',  'La pizza è buona. La pizza è buona perché è cucinata bene.'),
    ('degenerate',   ' '.join(['test'] * 20)),
]
for label, txt in demos:
    s = ngram_repetition_score(txt, n=3)
    print(f'{label:<14} score={s:.3f}  text={txt[:60]!r}')

clean prose    score=0.000  text='Il sole illumina il mondo, le stelle brillano nel cielo nott'
mild repeat    score=0.200  text='La pizza è buona. La pizza è buona perché è cucinata bene.'
degenerate     score=0.944  text='test test test test test test test test test test test test '


## Stage 2 — Language fidelity

After length, the highest-yield filter on machine-translated data is *did the translation actually finish*. Two checks:

1. **fastText langid** (`lid.176`) — Facebook's small 1MB model that tags 176 languages. Catches rows where English chunks survived translation.
2. **Italian function-word ratio** — A simpler heuristic: count the fraction of tokens that are Italian function words (`il, di, che, è, non, …`). Any fluent Italian text has 25–40% function words; English text has ~0%. This is a model-free fallback that complements langid.

In [4]:
# Function-word ratio: cheap, model-free language signal.
demos = [
    ('italian',    'Il gatto è sul tavolo e mangia il pesce mentre la luna brilla nel cielo.'),
    ('english',    'The cat eats fish under the moon while stars shine brightly above.'),
    ('mixed',      'The gatto is sul table and mangia the fish.'),
    ('formal_it',  'Vorremmo cordialmente invitarVi alla cerimonia che si terrà presso la nostra sede.'),
]
for label, txt in demos:
    r = function_word_ratio(txt)
    flag = 'OK' if r >= 0.05 else 'DROP'
    print(f'{label:<10} ratio={r:.3f}  [{flag}]   text={txt[:65]!r}')

italian    ratio=0.533  [OK]   text='Il gatto è sul tavolo e mangia il pesce mentre la luna brilla nel'
english    ratio=0.000  [DROP]   text='The cat eats fish under the moon while stars shine brightly above'
mixed      ratio=0.111  [OK]   text='The gatto is sul table and mangia the fish.'
formal_it  ratio=0.333  [OK]   text='Vorremmo cordialmente invitarVi alla cerimonia che si terrà press'


In [5]:
# Langid filter — gracefully skips if fasttext-wheel is not installed.
for r in fixture:
    verdict = langid_filter(r)
    if verdict is not None:
        print(f'{verdict:<30} {assistant_text(r)[:70]!r}')
print('(If you see no output and fasttext is not installed, the filter is a no-op — that\'s expected.)')

(If you see no output and fasttext is not installed, the filter is a no-op — that's expected.)


## Stage 3 — Translation-artifact patterns

Even after langid passes, rows can still be machine-translated junk that *happens* to be syntactically Italian. Two pattern-based filters catch the highest-precision tells:

- **English-artifact regexes**: untranslated filler (`I'm sorry`, `as an AI`), US-only units (`$`, `°F`, ZIP codes), AI-brand mentions (`ChatGPT`, `OpenAI`).
- **Italian calques**: literal translations that no native speaker would produce. The blocklist is intentionally conservative (precision over recall). The right way to extend it is *by sampling*: pull 200 surviving rows after stage 4, read them, and add to the list as you find new offenders.

Your job, as the course author, is to *grow this list* as a native speaker. The starter list catches the worst offenders, not all of them.

In [6]:
print(f'{"filter":<14} {"verdict":<48} text')
print('-' * 110)
for r in fixture:
    for fname, f in [('artifact', english_artifact_filter), ('calque', calque_filter)]:
        v = f(r)
        if v is not None:
            print(f'{fname:<14} {v:<48} {assistant_text(r)[:60]!r}')

filter         verdict                                          text
--------------------------------------------------------------------------------------------------------------
artifact       english_artifact:english_refusal_im_sorry        "I'm sorry, as an AI language model, I cannot answer that."
artifact       english_artifact:us_dollar_amount                'Una pizza media costa $25 nelle pizzerie di Manhattan.'
calque         calque:calque_modello_linguistico                'Come modello linguistico, non posso fornire opinioni persona'


## Stage 4 — Dedup & diversity

After per-row filters, the surviving rows likely have two remaining problems:

1. **Near-duplicates** — aggregations are duplicate-heavy. The same prompt template (e.g. *"Riassumi questo testo: …"*) repeats with different texts. MinHash + LSH gives O(N) approximate Jaccard on millions of rows.
2. **Task-family imbalance** — one or two prompt types may dominate the source. Without correcting for this, SFT will overfit to the dominant pattern. We fix it by embedding prompts with a multilingual encoder, clustering, and sampling uniformly across clusters.

The MinHash cell below runs if `datasketch` is installed.

In [7]:
# MinHash dedup demo on contrived near-duplicates.
try:
    from filters import minhash_dedup
    near_dup_demo = [
        row('Riassumi in due frasi il seguente testo: la pizza è popolare.',  'A.'),
        row('Riassumi in due frasi il seguente testo: la pizza è popolare!',  'B.'),  # near-dup
        row('Traduci la frase "the cat is on the table" in italiano.',         'C.'),
        row('Scrivi una breve poesia sull autunno e le foglie che cadono.',    'D.'),
    ]
    kept, dropped = minhash_dedup(near_dup_demo, threshold=0.8)
    print(f'before: {len(near_dup_demo)} rows')
    print(f'after:  {len(kept)} rows ({dropped} near-dup dropped)')
    for r in kept:
        print(f'  KEPT: {user_text(r)[:60]!r}')
except (ImportError, RuntimeError) as e:
    print(f'datasketch not installed ({e.__class__.__name__}); install with: pip install -r requirements.txt')

datasketch not installed (RuntimeError); install with: pip install -r requirements.txt


## End-to-end pipeline

Now compose stages 1–3 into a full cascade and run it on the fixture. The point of the report is that you can *see exactly what was dropped and why*. On a real run, you should sample 10 rows from each drop reason and read them — that's how you tune the thresholds, and it's how you find calques the regex missed.

In [8]:
filters = [
    empty_or_copy_filter, length_filter, repetition_filter,
    langid_filter, function_word_filter,
    english_artifact_filter, calque_filter,
]
kept, report = run_per_row_filters(fixture, filters)
print(report.summary())
print(f'\nsurvivors:')
for r in kept:
    print(f'  - {user_text(r)[:65]!r}')

total seen:           10
kept:                  3  ( 30.0%)
dropped:               7
by reason:
  user_too_short                                    3
  assistant_too_short                               2
  response_equals_prompt                            1
  english_artifact:english_refusal_im_sorry          1

survivors:
  - 'Riassumi questo testo in due frasi: il sole è una stella della no'
  - 'Scrivi una breve poesia sull autunno e le foglie dorate.'
  - 'Spiega cosa fa la funzione print() in Python.'


## Eval probes

Alongside the curation pipeline, this module ships `eval_probes.jsonl` — ~150 candidate Italian prompts across 15 task categories (factual QA, summarization, translation, creative writing, reasoning, code, roleplay, cultural, …) for you to curate down to a held-out evaluation set.

**Why a held-out eval set is non-negotiable for SFT on a new language:** English benchmarks (MMLU, IFEval, …) don't tell you whether your Italian SFT produced fluent Italian — they tell you that the model still has the knowledge to *answer English questions about Italian topics*, which is a different thing. The only way to know whether your model speaks fluent Italian is to ask it Italian questions and read the answers yourself. The probe set is what you ask.

In [9]:
import json
from collections import Counter

by_cat = Counter()
examples = []
with open('eval_probes.jsonl') as f:
    for line in f:
        p = json.loads(line)
        by_cat[p['category']] += 1
        examples.append(p)
print('coverage by category:')
for cat, n in sorted(by_cat.items(), key=lambda kv: -kv[1]):
    print(f'  {cat:<25s} {n}')
print(f'\ntotal: {sum(by_cat.values())} probes')
print(f'\nexample (category={examples[0]["category"]}):\n  prompt: {examples[0]["prompt"]}\n  notes:  {examples[0]["notes"]}')

coverage by category:
  factual_qa                15
  creative_writing          15
  italian_culture           15
  summarization             10
  translation               10
  explanation               10
  reasoning                 10
  code                      10
  instruction_following     10
  how_to                    10
  register                  10
  roleplay                  10
  refusal_or_ambiguity      5
  clarification             5
  open_ended                5

total: 150 probes

example (category=factual_qa):
  prompt: Chi ha scritto la Divina Commedia e in quale secolo è stata composta?
  notes:  Risposta attesa: Dante Alighieri, XIV secolo (1308-1320).


## Pushing the curated dataset to HuggingFace

Once `prepare.py` has produced your `italian-sft-curated.jsonl`, publishing is two commands. The `dataset_card.md` template ships with this module — fill in the `<N>`, `<date>`, and `<your-username>` placeholders before running.

```bash
# 1. Log in (once).
huggingface-cli login

# 2. Create the repo and upload.
huggingface-cli repo create italian-sft-curated --type dataset
huggingface-cli upload <your-username>/italian-sft-curated \
    italian-sft-curated.jsonl train.jsonl --repo-type=dataset
huggingface-cli upload <your-username>/italian-sft-curated \
    dataset_card.md README.md --repo-type=dataset
```

Pin a revision tag (`v1.0`) so anyone consuming the dataset can reproducibly load the exact bytes you trained on:

```bash
huggingface-cli repo tag <your-username>/italian-sft-curated v1.0
```

Then loading is:

```python
from datasets import load_dataset
ds = load_dataset('<your-username>/italian-sft-curated', revision='v1.0', split='train')
```

Pinning the revision is the difference between a reproducible course and a moving target.

## Recap — the takeaways worth keeping

1. **Quantity ≠ quality.** 100k high-precision rows beat 2M noisy ones in SFT. The interesting decision in this module is *what to throw away*, not what to keep.
2. **Cheap filters first.** Length, copy-bugs, repetition, langid, function-word ratio — all O(1) per row, all run before you pay for embeddings or LLM-judging.
3. **Pattern filters need maintenance.** The calque blocklist is a *native-speaker exercise*. Sample, read, extend.
4. **Look at the drop report.** The `report.json` artifact is the point. If your `assistant_too_short` count is 40% of the dataset, the threshold is wrong. If your `english_artifact:us_dollar_amount` count is huge, the upstream dataset is more US-centric than you thought.
5. **A held-out eval set is the only honest signal.** Build it yourself in the target language. English benchmarks don't measure Italian fluency.